# Лабораторная работа № 4
## Проектирование и сборка аналитической витрины

**Выполнили:** Попов Никита и Гордий Владимир  
**Группа:** ИВТ-241

### Выбранный вопрос из Project Brief

**Как различается количество аренд велосипедов в зависимости от сезона и погодных условий?**

**Источник:** `staging.parquet`, подготовленный в ЛР № 3 и сохранённый в S3.

**Метрики и срезы:** `rentals_count` — число аренд за час; `season` и `weathersit` — категории для группировки и сравнения.

**Граница анализа:** исходная детализация уже почасовая. Витрина не агрегирует и не фильтрует строки: она переименовывает и явно типизирует необходимые поля, сохраняя ключ и флаги качества из ЛР № 3.


## 1. Параметры проекта и S3

Входная и выходная логические даты совпадают. `source_ingested_on` — дата загрузки в S3, а `dteday` — дата почасового наблюдения в данных. Это разные значения.

Корень проекта находится поиском `pyproject.toml` и `.env.example` среди текущего каталога и его родителей, поэтому загрузка `.env` не зависит от того, запущен kernel из корня репозитория или из `notebooks/`.


In [1]:
from pathlib import Path
import os

import duckdb
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates = (current, *current.parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / ".env.example").is_file():
            return candidate
    raise FileNotFoundError(
        "Не удалось найти корень проекта: среди текущего каталога и его родителей "
        "нет одновременно pyproject.toml и .env.example."
    )


def read_env_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if not path.is_file():
        raise FileNotFoundError(
            f"Локальный .env не найден: {path}. Создайте его из .env.example; "
            "не добавляйте .env в Git."
        )
    for raw_line in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def sql_literal(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


PROJECT_ROOT = find_project_root()
ENV_PATH = PROJECT_ROOT / ".env"
env = read_env_file(ENV_PATH)

S3_ACCESS_KEY = env.get("S3_ACCESS_KEY", "s3admin")
S3_SECRET_KEY = env.get("S3_SECRET_KEY", "s3admin123")
S3_ENDPOINT = env.get("S3_ENDPOINT", "localhost:8333").strip()
S3_ENDPOINT = S3_ENDPOINT.removeprefix("http://").removeprefix("https://").rstrip("/")

DATASET_SLUG = "bike-sharing"
INGESTED_ON = "2026-10-18"
MART_NAME = "hourly_rentals"

STAGING_PATH = (
    f"s3://staging/{DATASET_SLUG}/"
    f"source_ingested_on={INGESTED_ON}/staging.parquet"
)
MART_PATH = (
    f"s3://mart/{DATASET_SLUG}/"
    f"source_ingested_on={INGESTED_ON}/{MART_NAME}.parquet"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STAGING_PATH:", STAGING_PATH)
print("MART_PATH:", MART_PATH)
print("MART_NAME:", MART_NAME)


PROJECT_ROOT: C:\itad-lab2
STAGING_PATH: s3://staging/bike-sharing/source_ingested_on=2026-10-18/staging.parquet
MART_PATH: s3://mart/bike-sharing/source_ingested_on=2026-10-18/hourly_rentals.parquet
MART_NAME: hourly_rentals


## 2. Подключение и проверка входного staging

Читаем Parquet непосредственно из S3 через DuckDB. До проектирования витрины проверяем фактическую схему, количество строк и наличие полей, которые понадобятся для выбранного вопроса.


In [2]:
con = duckdb.connect()

try:
    con.execute("LOAD httpfs")
except Exception:
    con.execute("INSTALL httpfs")
    con.execute("LOAD httpfs")

con.execute(f"SET s3_endpoint={sql_literal(S3_ENDPOINT)}")
con.execute(f"SET s3_access_key_id={sql_literal(S3_ACCESS_KEY)}")
con.execute(f"SET s3_secret_access_key={sql_literal(S3_SECRET_KEY)}")
con.execute("SET s3_region='us-east-1'")
con.execute("SET s3_url_style='path'")
con.execute("SET s3_use_ssl=false")

con.execute(
    f"CREATE OR REPLACE TEMP VIEW staging_data AS "
    f"SELECT * FROM read_parquet({sql_literal(STAGING_PATH)})"
)

staging_schema = con.execute("DESCRIBE staging_data").df()
staging_columns = set(staging_schema["column_name"].tolist())
staging_rows = int(con.execute("SELECT COUNT(*) FROM staging_data").fetchone()[0])

required_fields = {
    "dteday", "hr", "season", "weathersit", "yr", "mnth",
    "weekday", "workingday", "holiday", "temp", "atemp",
    "hum", "windspeed", "cnt", "casual", "registered",
}
missing_required_fields = sorted(required_fields - staging_columns)

print("DuckDB:", duckdb.__version__)
print("Входных строк в staging:", staging_rows)
print("Число полей staging:", len(staging_columns))
print("Отсутствующие обязательные поля:", missing_required_fields)
display(staging_schema)

if missing_required_fields:
    raise ValueError(
        "В staging отсутствуют поля, необходимые выбранной витрине: "
        + ", ".join(missing_required_fields)
    )


DuckDB: 1.1.3
Входных строк в staging: 17379
Число полей staging: 20
Отсутствующие обязательные поля: []


,column_name,column_type,null,key,default,extra
0,instant,INTEGER,YES,None,None,None
1,dteday,DATE,YES,None,None,None
2,season,INTEGER,YES,None,None,None
3,yr,INTEGER,YES,None,None,None
4,mnth,INTEGER,YES,None,None,None
5,hr,INTEGER,YES,None,None,None
6,holiday,INTEGER,YES,None,None,None
7,weekday,INTEGER,YES,None,None,None
8,workingday,INTEGER,YES,None,None,None
9,weathersit,INTEGER,YES,None,None,None


In [3]:
# В ЛР № 3 были добавлены эти два поля. Если они присутствуют,
# переносим их значения в витрину без подмены NULL/пустых значений.
QUALITY_FIELDS_PRESENT = {
    "quality_flag_ids": "quality_flag_ids" in staging_columns,
    "quality_flag_reason": "quality_flag_reason" in staging_columns,
}

quality_flags_availability = pd.DataFrame(
    [
        {"field": field, "present_in_staging": is_present}
        for field, is_present in QUALITY_FIELDS_PRESENT.items()
    ]
)
quality_flags_availability


,field,present_in_staging
0,quality_flag_ids,True
1,quality_flag_reason,True


## 3. Гранулярность, ключ и меры

**Одна строка `hourly_rentals` представляет одно почасовое наблюдение спроса в указанную дату.**

**Естественный составной ключ:** `dteday + hr` — дата наблюдения и час суток. В ЛР № 3 повторов по этому ключу не было; ниже ключ проверяется повторно и на входе, и на выходной витрине. Мы не создаём искусственный идентификатор и не используем `DISTINCT`.

**Мера:** `rentals_count` — количество аренд за этот час, перенесено из `cnt` без изменения смысла. Количество аренд за непересекающиеся часы можно суммировать. Средние значения нельзя суммировать; при последующем укрупнении среднее следует рассчитывать по исходным наблюдениям или как взвешенное среднее.

**Срезы:** `season`, `weathersit`; дополнительные календарные поля позволяют затем уточнять сравнения.

Модель — одна широкая таблица: все поля относятся к одной и той же почасовой записи, поэтому отдельные таблицы измерений и `JOIN` для ответа на вопрос не требуются.


In [4]:
# Проверка ключа и полноты основных полей во входном staging.
staging_key_check = con.execute(
    """
    SELECT
        COUNT(*) FILTER (WHERE dteday IS NULL) AS null_dteday,
        COUNT(*) FILTER (WHERE hr IS NULL) AS null_hr,
        COUNT(*) FILTER (
            WHERE dteday IS NOT NULL AND hr IS NOT NULL
        ) AS rows_with_complete_key
    FROM staging_data
    """
).df()

staging_duplicate_check = con.execute(
    """
    SELECT
        COUNT(*) AS duplicate_key_groups,
        COALESCE(SUM(rows_per_key - 1), 0) AS duplicate_extra_rows
    FROM (
        SELECT dteday, hr, COUNT(*) AS rows_per_key
        FROM staging_data
        GROUP BY dteday, hr
        HAVING COUNT(*) > 1
    )
    """
).df()

display(staging_key_check)
display(staging_duplicate_check)


,null_dteday,null_hr,rows_with_complete_key
0,0,0,17379


,duplicate_key_groups,duplicate_extra_rows
0,0,0.0


## 4. Сборка витрины

Преобразование намеренно не применяет `WHERE`, `DISTINCT`, агрегацию или `JOIN`: ни одна строка `staging` не исключается и не размножается. Из `cnt`, `casual` и `registered` создаются имена, лучше выражающие смысл мер в витрине. Типы перечислены явно.

Если флаги качества из ЛР № 3 присутствуют во входном `staging`, их значения переносятся в `hourly_rentals`. Если они отсутствуют, соответствующие выходные поля остаются `NULL`, а ячейка выше показывает это ограничение.


In [5]:
quality_flag_ids_expr = (
    "CAST(quality_flag_ids AS VARCHAR)"
    if QUALITY_FIELDS_PRESENT["quality_flag_ids"]
    else "CAST(NULL AS VARCHAR)"
)
quality_flag_reason_expr = (
    "CAST(quality_flag_reason AS VARCHAR)"
    if QUALITY_FIELDS_PRESENT["quality_flag_reason"]
    else "CAST(NULL AS VARCHAR)"
)

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW mart_hourly_rentals AS
    SELECT
        CAST(dteday AS DATE) AS dteday,
        CAST(hr AS INTEGER) AS hr,
        CAST(season AS INTEGER) AS season,
        CAST(weathersit AS INTEGER) AS weathersit,
        CAST(yr AS INTEGER) AS yr,
        CAST(mnth AS INTEGER) AS mnth,
        CAST(weekday AS INTEGER) AS weekday,
        CAST(workingday AS INTEGER) AS workingday,
        CAST(holiday AS INTEGER) AS holiday,
        CAST(temp AS DOUBLE) AS temp,
        CAST(atemp AS DOUBLE) AS atemp,
        CAST(hum AS DOUBLE) AS hum,
        CAST(windspeed AS DOUBLE) AS windspeed,
        CAST(cnt AS BIGINT) AS rentals_count,
        CAST(casual AS BIGINT) AS casual_rentals_count,
        CAST(registered AS BIGINT) AS registered_rentals_count,
        {quality_flag_ids_expr} AS quality_flag_ids,
        {quality_flag_reason_expr} AS quality_flag_reason
    FROM staging_data
    """
)

mart_view_count = int(
    con.execute("SELECT COUNT(*) FROM mart_hourly_rentals").fetchone()[0]
)
mart_view_schema = con.execute("DESCRIBE mart_hourly_rentals").df()

print("Строк во входном staging:", staging_rows)
print("Строк в представлении витрины:", mart_view_count)
print("Строк исключено по условиям фильтрации: 0 (фильтрации нет)")
display(mart_view_schema)


Строк во входном staging: 17379
Строк в представлении витрины: 17379
Строк исключено по условиям фильтрации: 0 (фильтрации нет)


,column_name,column_type,null,key,default,extra
0,dteday,DATE,YES,None,None,None
1,hr,INTEGER,YES,None,None,None
2,season,INTEGER,YES,None,None,None
3,weathersit,INTEGER,YES,None,None,None
4,yr,INTEGER,YES,None,None,None
5,mnth,INTEGER,YES,None,None,None
6,weekday,INTEGER,YES,None,None,None
7,workingday,INTEGER,YES,None,None,None
8,holiday,INTEGER,YES,None,None,None
9,temp,DOUBLE,YES,None,None,None


In [6]:
# Материализуем витрину в S3. Повторный запуск перезаписывает предыдущий Parquet.
con.execute(
    f"""
    COPY (
        SELECT
            dteday, hr, season, weathersit, yr, mnth, weekday,
            workingday, holiday, temp, atemp, hum, windspeed,
            rentals_count, casual_rentals_count, registered_rentals_count,
            quality_flag_ids, quality_flag_reason
        FROM mart_hourly_rentals
    )
    TO {sql_literal(MART_PATH)}
    (FORMAT PARQUET, OVERWRITE_OR_IGNORE TRUE)
    """
)

print("Витрина сохранена:")
print(MART_PATH)


Витрина сохранена:
s3://mart/bike-sharing/source_ingested_on=2026-10-18/hourly_rentals.parquet


## 5. Проверка сохранённой витрины

Проверки выполняются не только на временном представлении: сначала повторно читаем итоговый Parquet именно из S3-бакета `mart`, затем сравниваем его с входным `staging`.


In [7]:
# Читаем физическую схему Parquet без автоматического
# добавления виртуальных полей из Hive-партиций.
con.execute(
    f"CREATE OR REPLACE TEMP VIEW saved_mart AS "
    f"SELECT * FROM read_parquet("
    f"{sql_literal(MART_PATH)}, hive_partitioning = false)"
)

mart_rows = int(
    con.execute("SELECT COUNT(*) FROM saved_mart").fetchone()[0]
)
mart_schema = con.execute("DESCRIBE saved_mart").df()

row_balance = pd.DataFrame([{
    "staging_rows": staging_rows,
    "mart_rows": mart_rows,
    "excluded_rows_by_design": 0,
    "staging_equals_mart_plus_excluded": staging_rows == mart_rows + 0,
    "row_count_preserved": staging_rows == mart_rows,
}])

display(row_balance)
display(mart_schema)

mart_rows = int(con.execute("SELECT COUNT(*) FROM saved_mart").fetchone()[0])
mart_schema = con.execute("DESCRIBE saved_mart").df()

row_balance = pd.DataFrame([{
    "staging_rows": staging_rows,
    "mart_rows": mart_rows,
    "excluded_rows_by_design": 0,
    "staging_equals_mart_plus_excluded": staging_rows == mart_rows + 0,
    "row_count_preserved": staging_rows == mart_rows,
}])

display(row_balance)
display(mart_schema)


,staging_rows,mart_rows,excluded_rows_by_design,staging_equals_mart_plus_excluded,row_count_preserved
0,17379,17379,0,True,True


,column_name,column_type,null,key,default,extra
0,dteday,DATE,YES,None,None,None
1,hr,INTEGER,YES,None,None,None
2,season,INTEGER,YES,None,None,None
3,weathersit,INTEGER,YES,None,None,None
4,yr,INTEGER,YES,None,None,None
5,mnth,INTEGER,YES,None,None,None
6,weekday,INTEGER,YES,None,None,None
7,workingday,INTEGER,YES,None,None,None
8,holiday,INTEGER,YES,None,None,None
9,temp,DOUBLE,YES,None,None,None


,staging_rows,mart_rows,excluded_rows_by_design,staging_equals_mart_plus_excluded,row_count_preserved
0,17379,17379,0,True,True


,column_name,column_type,null,key,default,extra
0,dteday,DATE,YES,None,None,None
1,hr,INTEGER,YES,None,None,None
2,season,INTEGER,YES,None,None,None
3,weathersit,INTEGER,YES,None,None,None
4,yr,INTEGER,YES,None,None,None
5,mnth,INTEGER,YES,None,None,None
6,weekday,INTEGER,YES,None,None,None
7,workingday,INTEGER,YES,None,None,None
8,holiday,INTEGER,YES,None,None,None
9,temp,DOUBLE,YES,None,None,None


In [8]:
# Проверка состава и типов всех полей сохранённой витрины.
expected_schema = [
    ("dteday", "DATE"),
    ("hr", "INTEGER"),
    ("season", "INTEGER"),
    ("weathersit", "INTEGER"),
    ("yr", "INTEGER"),
    ("mnth", "INTEGER"),
    ("weekday", "INTEGER"),
    ("workingday", "INTEGER"),
    ("holiday", "INTEGER"),
    ("temp", "DOUBLE"),
    ("atemp", "DOUBLE"),
    ("hum", "DOUBLE"),
    ("windspeed", "DOUBLE"),
    ("rentals_count", "BIGINT"),
    ("casual_rentals_count", "BIGINT"),
    ("registered_rentals_count", "BIGINT"),
    ("quality_flag_ids", "VARCHAR"),
    ("quality_flag_reason", "VARCHAR"),
]

actual_schema = [
    (row["column_name"], row["column_type"].upper())
    for _, row in mart_schema.iterrows()
]
schema_matches = actual_schema == expected_schema

schema_check = pd.DataFrame([
    {
        "position": index + 1,
        "expected_column": expected_name,
        "expected_type": expected_type,
        "actual_column": actual_schema[index][0] if index < len(actual_schema) else None,
        "actual_type": actual_schema[index][1] if index < len(actual_schema) else None,
        "matches": (
            index < len(actual_schema)
            and actual_schema[index] == (expected_name, expected_type)
        ),
    }
    for index, (expected_name, expected_type) in enumerate(expected_schema)
])

print("Схема полностью соответствует модели:", schema_matches)
display(schema_check)


Схема полностью соответствует модели: True


,position,expected_column,expected_type,actual_column,actual_type,matches
0,1,dteday,DATE,dteday,DATE,True
1,2,hr,INTEGER,hr,INTEGER,True
2,3,season,INTEGER,season,INTEGER,True
3,4,weathersit,INTEGER,weathersit,INTEGER,True
4,5,yr,INTEGER,yr,INTEGER,True
5,6,mnth,INTEGER,mnth,INTEGER,True
6,7,weekday,INTEGER,weekday,INTEGER,True
7,8,workingday,INTEGER,workingday,INTEGER,True
8,9,holiday,INTEGER,holiday,INTEGER,True
9,10,temp,DOUBLE,temp,DOUBLE,True


In [9]:
print("Ожидается колонок:", len(expected_schema))
print("Фактически колонок:", len(actual_schema))

print("Дополнительные поля:")
print([
    field for field in actual_schema
    if field not in expected_schema
])

print("Фактическая схема:")
print(actual_schema)

Ожидается колонок: 18
Фактически колонок: 18
Дополнительные поля:
[]
Фактическая схема:
[('dteday', 'DATE'), ('hr', 'INTEGER'), ('season', 'INTEGER'), ('weathersit', 'INTEGER'), ('yr', 'INTEGER'), ('mnth', 'INTEGER'), ('weekday', 'INTEGER'), ('workingday', 'INTEGER'), ('holiday', 'INTEGER'), ('temp', 'DOUBLE'), ('atemp', 'DOUBLE'), ('hum', 'DOUBLE'), ('windspeed', 'DOUBLE'), ('rentals_count', 'BIGINT'), ('casual_rentals_count', 'BIGINT'), ('registered_rentals_count', 'BIGINT'), ('quality_flag_ids', 'VARCHAR'), ('quality_flag_reason', 'VARCHAR')]


In [10]:
# Проверка составного ключа dteday + hr в прочитанном Parquet.
mart_key_check = con.execute(
    """
    SELECT
        COUNT(*) FILTER (WHERE dteday IS NULL) AS null_dteday,
        COUNT(*) FILTER (WHERE hr IS NULL) AS null_hr
    FROM saved_mart
    """
).df()

mart_duplicate_check = con.execute(
    """
    SELECT
        COUNT(*) AS duplicate_key_groups,
        COALESCE(SUM(rows_per_key - 1), 0) AS duplicate_extra_rows
    FROM (
        SELECT dteday, hr, COUNT(*) AS rows_per_key
        FROM saved_mart
        GROUP BY dteday, hr
        HAVING COUNT(*) > 1
    )
    """
).df()

display(mart_key_check)
display(mart_duplicate_check)


,null_dteday,null_hr
0,0,0


,duplicate_key_groups,duplicate_extra_rows
0,0,0.0


In [11]:
# Проверяем обязательные поля, меры, диапазоны и согласованность cnt = casual + registered.
measure_and_range_check = con.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE dteday IS NULL OR hr IS NULL OR season IS NULL
               OR weathersit IS NULL OR rentals_count IS NULL
        ) AS rows_with_null_required_fields,
        COUNT(*) FILTER (
            WHERE rentals_count IS NULL
               OR casual_rentals_count IS NULL
               OR registered_rentals_count IS NULL
        ) AS rows_with_null_measures,
        MIN(rentals_count) AS min_rentals_count,
        MAX(rentals_count) AS max_rentals_count,
        COUNT(*) FILTER (WHERE rentals_count < 0) AS negative_rentals_rows,
        MIN(hr) AS min_hr,
        MAX(hr) AS max_hr,
        COUNT(*) FILTER (WHERE hr < 0 OR hr > 23) AS invalid_hr_rows,
        COUNT(*) FILTER (WHERE season NOT IN (1, 2, 3, 4)) AS invalid_season_rows,
        COUNT(*) FILTER (WHERE weathersit NOT IN (1, 2, 3, 4)) AS invalid_weather_rows,
        COUNT(*) FILTER (
            WHERE casual_rentals_count + registered_rentals_count <> rentals_count
        ) AS component_sum_mismatch_rows
    FROM saved_mart
    """
).df()

measure_and_range_check


,rows_with_null_required_fields,rows_with_null_measures,min_rentals_count,max_rentals_count,negative_rentals_rows,min_hr,max_hr,invalid_hr_rows,invalid_season_rows,invalid_weather_rows,component_sum_mismatch_rows
0,0,0,1,977,0,0,23,0,0,0,0


In [12]:
# Сверяем количество строк с флагами и сами флаги между staging и mart.
if QUALITY_FIELDS_PRESENT["quality_flag_ids"] and QUALITY_FIELDS_PRESENT["quality_flag_reason"]:
    flags_summary = con.execute(
        """
        SELECT
            (SELECT COUNT(*) FROM staging_data
             WHERE COALESCE(quality_flag_ids, '') <> ''
                OR COALESCE(quality_flag_reason, '') <> '') AS staging_flagged_rows,
            (SELECT COUNT(*) FROM saved_mart
             WHERE COALESCE(quality_flag_ids, '') <> ''
                OR COALESCE(quality_flag_reason, '') <> '') AS mart_flagged_rows,
            (SELECT COUNT(*) FROM (
                SELECT dteday, hr, quality_flag_ids, quality_flag_reason FROM staging_data
                EXCEPT
                SELECT dteday, hr, quality_flag_ids, quality_flag_reason FROM saved_mart
            )) AS source_flag_rows_missing_or_changed,
            (SELECT COUNT(*) FROM (
                SELECT dteday, hr, quality_flag_ids, quality_flag_reason FROM saved_mart
                EXCEPT
                SELECT dteday, hr, quality_flag_ids, quality_flag_reason FROM staging_data
            )) AS unexpected_mart_flag_rows
        """
    ).df()
else:
    flags_summary = pd.DataFrame([{
        "staging_flagged_rows": None,
        "mart_flagged_rows": None,
        "source_flag_rows_missing_or_changed": None,
        "unexpected_mart_flag_rows": None,
        "note": "В staging отсутствует одно или оба поля флагов ЛР № 3; проверьте ограничение выше.",
    }])

flags_summary


,staging_flagged_rows,mart_flagged_rows,source_flag_rows_missing_or_changed,unexpected_mart_flag_rows
0,0,0,0,0


### Короткий запрос по выбранному аналитическому вопросу

Группировка по `season` и `weathersit` проверяет, что витрина содержит нужные срезы и меру среднего количества аренд за наблюдаемый час. Это демонстрация пригодности модели, а не полный анализ и не причинно-следственный вывод.


In [13]:
season_weather_slice = con.execute(
    """
    SELECT
        season,
        weathersit,
        COUNT(*) AS hourly_observations,
        ROUND(AVG(rentals_count), 2) AS avg_rentals_per_observed_hour
    FROM saved_mart
    GROUP BY season, weathersit
    ORDER BY season, weathersit
    """
).df()

season_weather_slice


,season,weathersit,hourly_observations,avg_rentals_per_observed_hour
0,1,1,2665,122.06
1,1,2,1205,103.07
2,1,3,369,58.65
3,1,4,3,74.33
4,2,1,2859,232.41
5,2,2,1144,181.26
6,2,3,406,115.18
7,3,1,3280,244.49
8,3,2,947,224.85
9,3,3,269,171.96


In [14]:
# Сводная самопроверка. PASS ставится только по фактическим результатам этого запуска.
key_nulls = int(mart_key_check.loc[0, "null_dteday"]) + int(mart_key_check.loc[0, "null_hr"])
duplicate_groups = int(mart_duplicate_check.loc[0, "duplicate_key_groups"])
null_required = int(measure_and_range_check.loc[0, "rows_with_null_required_fields"])
null_measures = int(measure_and_range_check.loc[0, "rows_with_null_measures"])
negative_rentals = int(measure_and_range_check.loc[0, "negative_rentals_rows"])
invalid_hr = int(measure_and_range_check.loc[0, "invalid_hr_rows"])
invalid_season = int(measure_and_range_check.loc[0, "invalid_season_rows"])
invalid_weather = int(measure_and_range_check.loc[0, "invalid_weather_rows"])
sum_mismatches = int(measure_and_range_check.loc[0, "component_sum_mismatch_rows"])

if QUALITY_FIELDS_PRESENT["quality_flag_ids"] and QUALITY_FIELDS_PRESENT["quality_flag_reason"]:
    flag_row = flags_summary.loc[0]
    flags_preserved = (
        int(flag_row["staging_flagged_rows"]) == int(flag_row["mart_flagged_rows"])
        and int(flag_row["source_flag_rows_missing_or_changed"]) == 0
        and int(flag_row["unexpected_mart_flag_rows"]) == 0
    )
else:
    flags_preserved = None

summary_checks = [
    ("Количество строк staging = mart", staging_rows == mart_rows),
    ("Порядок и типы выходных полей соответствуют модели", schema_matches),
    ("Компоненты ключа не содержат NULL", key_nulls == 0),
    ("Повторов dteday + hr нет", duplicate_groups == 0),
    ("Обязательные поля заполнены", null_required == 0),
    ("Меры заполнены", null_measures == 0),
    ("Количество аренд неотрицательно", negative_rentals == 0),
    ("Час находится в диапазоне 0..23", invalid_hr == 0),
    ("Коды сезона входят в {1,2,3,4}", invalid_season == 0),
    ("Коды погоды входят в {1,2,3,4}", invalid_weather == 0),
    ("casual + registered = rentals_count", sum_mismatches == 0),
]
if flags_preserved is not None:
    summary_checks.append(("Флаги качества из staging сохранены", flags_preserved))

checks_summary = pd.DataFrame([
    {"check": name, "result": "PASS" if result else "FAIL"}
    for name, result in summary_checks
])
display(checks_summary)

print("Входной staging:", STAGING_PATH)
print("Сохранённая и повторно прочитанная витрина:", MART_PATH)
print("Итоговые строки — staging:", staging_rows, "| mart:", mart_rows)
print("Исключено строк по модели: 0; фильтрация и агрегация при построении не выполнялись.")


,check,result
0,Количество строк staging = mart,PASS
1,Порядок и типы выходных полей соответствуют мо...,PASS
2,Компоненты ключа не содержат NULL,PASS
3,Повторов dteday + hr нет,PASS
4,Обязательные поля заполнены,PASS
5,Меры заполнены,PASS
6,Количество аренд неотрицательно,PASS
7,Час находится в диапазоне 0..23,PASS
8,"Коды сезона входят в {1,2,3,4}",PASS
9,"Коды погоды входят в {1,2,3,4}",PASS


Входной staging: s3://staging/bike-sharing/source_ingested_on=2026-10-18/staging.parquet
Сохранённая и повторно прочитанная витрина: s3://mart/bike-sharing/source_ingested_on=2026-10-18/hourly_rentals.parquet
Итоговые строки — staging: 17379 | mart: 17379
Исключено строк по модели: 0; фильтрация и агрегация при построении не выполнялись.
